In [ ]:
# Notebook 08 - Location Opportunity Scoring Model
# Convert engineered signals into a final decision score

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Upload opportunity features

from google.colab import files

uploaded = files.upload()

opportunity = pd.read_csv(
    "location_opportunity_features.csv"
)

print("Shape:", opportunity.shape)
print("\nColumns:")
print(opportunity.columns.tolist())

Saving location_opportunity_features.csv to location_opportunity_features.csv
Shape: (3232, 36)

Columns:
['id', 'brand', 'latitude', 'longitude', 'city', 'state', 'location_lat', 'location_lon', 'Total_Dark_Stores', 'Competing_Brands', 'Competition_Intensity', 'Same_Brand_Stores', 'Competitor_Stores', 'Competitor_Ratio', 'Brand_Share', 'market_key', 'geo_lat', 'geo_lon', 'geo_key', 'assigned_market', 'market_distance_km', 'assignment_quality', 'Population', 'Households', 'Literacy_Rate', 'Workforce_Rate', 'Average_Household_Size', 'Retail_Demand_Index', 'Demand_Strength', 'Competition_Risk', 'Competitor_Pressure', 'Local_Brand_Strength', 'Market_Density', 'Demand_Gap', 'Expansion_Headroom', 'Market_Control']


In [ ]:
# Check scoring features

score_features = [
    "Demand_Strength",
    "Competition_Risk",
    "Competitor_Pressure",
    "Local_Brand_Strength",
    "Market_Density",
    "Expansion_Headroom"
]

print(
    opportunity[score_features]
    .describe()
    .round(3)
)

print("\nMissing values:")
print(
    opportunity[score_features]
    .isna()
    .sum()
)

       Demand_Strength  Competition_Risk  Competitor_Pressure  \
count         3232.000          3232.000             3232.000   
mean             0.678             0.339                0.662   
std              0.270             0.244                0.178   
min              0.000             0.000                0.000   
25%              0.450             0.144                0.574   
50%              0.672             0.281                0.687   
75%              0.913             0.510                0.776   
max              1.000             1.000                1.000   

       Local_Brand_Strength  Market_Density  Expansion_Headroom  
count              3232.000        3232.000            3232.000  
mean                  0.338           0.335               0.221  
std                   0.178           0.245               0.135  
min                   0.000           0.000               0.000  
25%                   0.224           0.138               0.131  
50%               

In [ ]:
# Define scoring components

model_features = pd.DataFrame({
    "Feature": [
        "Demand_Strength",
        "Competition_Risk",
        "Competitor_Pressure",
        "Local_Brand_Strength",
        "Market_Density"
    ],

    "Direction": [
        "Positive",
        "Negative",
        "Negative",
        "Positive",
        "Negative"
    ],

    "Meaning": [
        "Underlying market demand",
        "Overall competitive intensity",
        "Share controlled by competitors",
        "Existing local brand position",
        "Existing dark-store concentration"
    ]
})

model_features

,Feature,Direction,Meaning
0,Demand_Strength,Positive,Underlying market demand
1,Competition_Risk,Negative,Overall competitive intensity
2,Competitor_Pressure,Negative,Share controlled by competitors
3,Local_Brand_Strength,Positive,Existing local brand position
4,Market_Density,Negative,Existing dark-store concentration


In [ ]:
# Convert risks into opportunity signals

opportunity["Competition_Opportunity"] = (
    1 - opportunity["Competition_Risk"]
)

opportunity["Pressure_Opportunity"] = (
    1 - opportunity["Competitor_Pressure"]
)

opportunity["Density_Opportunity"] = (
    1 - opportunity["Market_Density"]
)

In [ ]:
##demand 40% — expansion needs customers
#Competition 20% — avoid heavily contested markets
#competitor Pressure 15% — reward whitespace
#Brand Strength 15% — captures existing local advantage
#Density 10% — penalizes saturation


# Baseline model weights

weights = {
    "Demand_Strength": 0.40,
    "Competition_Opportunity": 0.20,
    "Pressure_Opportunity": 0.15,
    "Local_Brand_Strength": 0.15,
    "Density_Opportunity": 0.10
}

print("Total weight:", sum(weights.values()))

Total weight: 1.0


In [ ]:
# Calculate final opportunity score

opportunity["Opportunity_Score"] = 100 * (

    opportunity["Demand_Strength"]
    * weights["Demand_Strength"]

    + opportunity["Competition_Opportunity"]
    * weights["Competition_Opportunity"]

    + opportunity["Pressure_Opportunity"]
    * weights["Pressure_Opportunity"]

    + opportunity["Local_Brand_Strength"]
    * weights["Local_Brand_Strength"]

    + opportunity["Density_Opportunity"]
    * weights["Density_Opportunity"]
)

print(
    opportunity["Opportunity_Score"]
    .describe()
    .round(2)
)

count    3232.00
mean       57.13
std        13.14
min        26.45
25%        47.20
50%        58.26
75%        67.25
max       100.00
Name: Opportunity_Score, dtype: float64


In [ ]:
# Test alternative business strategies

scenarios = {

    "Balanced": {
        "Demand_Strength": 0.40,
        "Competition_Opportunity": 0.20,
        "Pressure_Opportunity": 0.15,
        "Local_Brand_Strength": 0.15,
        "Density_Opportunity": 0.10
    },

    "Demand_Focused": {
        "Demand_Strength": 0.55,
        "Competition_Opportunity": 0.15,
        "Pressure_Opportunity": 0.10,
        "Local_Brand_Strength": 0.10,
        "Density_Opportunity": 0.10
    },

    "Low_Competition": {
        "Demand_Strength": 0.30,
        "Competition_Opportunity": 0.25,
        "Pressure_Opportunity": 0.25,
        "Local_Brand_Strength": 0.10,
        "Density_Opportunity": 0.10
    }
}

In [ ]:
# Calculate score under each strategy

for scenario, w in scenarios.items():

    opportunity[scenario] = 100 * (
        opportunity["Demand_Strength"] * w["Demand_Strength"]
        + opportunity["Competition_Opportunity"] * w["Competition_Opportunity"]
        + opportunity["Pressure_Opportunity"] * w["Pressure_Opportunity"]
        + opportunity["Local_Brand_Strength"] * w["Local_Brand_Strength"]
        + opportunity["Density_Opportunity"] * w["Density_Opportunity"]
    )

opportunity[
    [
        "Opportunity_Score",
        "Balanced",
        "Demand_Focused",
        "Low_Competition"
    ]
].describe().round(2)

,Opportunity_Score,Balanced,Demand_Focused,Low_Competition
count,3232.00,3232.00,3232.00,3232.00
mean,57.13,57.13,60.61,55.35
std,13.14,13.14,15.23,12.86
min,26.45,26.45,23.44,24.44
25%,47.20,47.20,48.26,46.58
50%,58.26,58.26,63.08,55.01
75%,67.25,67.25,73.45,64.03
max,100.00,100.00,100.00,100.00


In [ ]:
# Create market opportunity ranking

market_scores = (
    opportunity
    .groupby("assigned_market", as_index=False)
    .agg(
        Opportunity_Score=("Opportunity_Score", "mean"),
        Balanced=("Balanced", "mean"),
        Demand_Focused=("Demand_Focused", "mean"),
        Low_Competition=("Low_Competition", "mean"),
        Demand_Strength=("Demand_Strength", "mean"),
        Competition_Risk=("Competition_Risk", "mean"),
        Population=("Population", "mean"),
        Existing_Stores=("id", "count")
    )
)

market_scores["Rank"] = (
    market_scores["Opportunity_Score"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

market_scores = market_scores.sort_values("Rank")

market_scores.head(15)

,assigned_market,Opportunity_Score,Balanced,Demand_Focused,Low_Competition,Demand_Strength,Competition_Risk,Population,Existing_Stores,Rank
7,coimbatore,74.992844,74.992844,80.214877,72.350008,0.908019,0.104709,3458045.0,26,1
46,vellore,74.142026,74.142026,76.558482,73.402457,0.785377,0.027159,3936331.0,7,2
27,madurai,73.992670,73.992670,77.816620,72.281403,0.841981,0.077736,3038252.0,9,3
32,palakkad,73.530663,73.530663,74.781580,73.525657,0.735849,0.015970,2809934.0,5,4
30,nagpur,73.052309,73.052309,78.430708,70.165420,0.903302,0.175774,4653570.0,35,5
42,surat,72.307090,72.307090,78.175329,69.293834,0.903302,0.128353,6081322.0,37,6
31,nashik,70.628958,70.628958,76.295004,67.693318,0.882075,0.158972,6107187.0,21,7
8,davanagere,70.617887,70.617887,67.634148,73.194393,0.551887,0.014259,1945497.0,4,8
37,pune,68.763810,68.763810,76.260833,64.453304,0.945755,0.279043,9429408.0,188,9
22,kolkata,67.843138,67.843138,74.346154,64.048536,0.905660,0.311577,4496694.0,181,10


In [ ]:
# Compare scenario rankings

market_scores["Rank_Balanced"] = (
    market_scores["Balanced"]
    .rank(ascending=False, method="dense")
)

market_scores["Rank_Demand"] = (
    market_scores["Demand_Focused"]
    .rank(ascending=False, method="dense")
)

market_scores["Rank_LowCompetition"] = (
    market_scores["Low_Competition"]
    .rank(ascending=False, method="dense")
)

market_scores["Rank_Stability"] = (
    market_scores[
        [
            "Rank_Balanced",
            "Rank_Demand",
            "Rank_LowCompetition"
        ]
    ]
    .std(axis=1)
)

market_scores[
    [
        "assigned_market",
        "Opportunity_Score",
        "Rank_Balanced",
        "Rank_Demand",
        "Rank_LowCompetition",
        "Rank_Stability"
    ]
].head(15)

,assigned_market,Opportunity_Score,Rank_Balanced,Rank_Demand,Rank_LowCompetition,Rank_Stability
7,coimbatore,74.992844,1.0,1.0,4.0,1.732051
46,vellore,74.142026,2.0,5.0,2.0,1.732051
27,madurai,73.992670,3.0,4.0,5.0,1.000000
32,palakkad,73.530663,4.0,8.0,1.0,3.511885
30,nagpur,73.052309,5.0,2.0,7.0,2.516611
42,surat,72.307090,6.0,3.0,8.0,2.516611
31,nashik,70.628958,7.0,6.0,10.0,2.081666
8,davanagere,70.617887,8.0,16.0,3.0,6.557439
37,pune,68.763810,9.0,7.0,15.0,4.163332
22,kolkata,67.843138,10.0,9.0,17.0,4.358899


In [ ]:
# Top opportunity markets

top_markets = (
    market_scores
    .sort_values("Opportunity_Score", ascending=False)
    .head(15)
    .copy()
)

fig = px.scatter(
    top_markets,

    x="Rank_Stability",
    y="Opportunity_Score",

    size="Population",
    color="Demand_Strength",

    text="assigned_market",

    hover_name="assigned_market",

    hover_data={
        "Rank_Balanced": ":.0f",
        "Rank_Demand": ":.0f",
        "Rank_LowCompetition": ":.0f",
        "Rank_Stability": ":.2f",
        "Opportunity_Score": ":.1f",
        "Population": ":,.0f"
    },

    color_continuous_scale=[
        "#D8EDE8",
        "#55B6A9",
        "#2878A5",
        "#313695"
    ],

    size_max=45
)

fig.update_traces(
    textposition="top center",

    marker=dict(
        opacity=0.78,
        line=dict(
            color="white",
            width=1.5
        )
    )
)

fig.add_annotation(
    x=0.02,
    y=0.98,

    xref="paper",
    yref="paper",

    text=(
        "<b>ROBUST OPPORTUNITIES</b><br>"
        "High Score • Stable Ranking"
    ),

    showarrow=False,

    bgcolor="rgba(85,182,169,0.12)",
    bordercolor="#55B6A9",
    borderpad=7
)

fig.update_layout(

    title=dict(
        text=(
            "<b>WHICH EXPANSION MARKETS REMAIN STRONG UNDER DIFFERENT STRATEGIES?</b><br>"
            "<span style='font-size:13px;color:#777777'>"
            "Opportunity strength tested against changes in model priorities"
            "</span>"
        ),

        x=0.5,
        xanchor="center",

        font=dict(
            size=21,
            color="#263B5E"
        )
    ),

    xaxis=dict(
        title="Ranking Instability →  Lower is Better",
        gridcolor="#EEEEEE"
    ),

    yaxis=dict(
        title="Location Opportunity Score →",
        gridcolor="#EEEEEE"
    ),

    coloraxis_colorbar=dict(
        title="Demand<br>Strength"
    ),

    plot_bgcolor="#FCFCFC",
    paper_bgcolor="white",

    width=1050,
    height=650,

    margin=dict(
        t=125,
        l=90,
        r=130,
        b=80
    )
)

fig.show()

In [ ]:
# Save Notebook 8 outputs

market_scores.to_csv(
    "market_opportunity_scores.csv",
    index=False
)

opportunity.to_csv(
    "scored_location_opportunities.csv",
    index=False
)

print("Notebook 8 outputs saved.")

Notebook 8 outputs saved.


In [ ]:
# Download Notebook 8 outputs

from google.colab import files

files.download("market_opportunity_scores.csv")
files.download("scored_location_opportunities.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>